# Лабораторна робота — Частина 1
## Аналіз VHI-індексу для регіонів України

### Встановлення залежностей
Перед початком роботи встановіть необхідні бібліотеки:


```bash
python -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

### Пункт 2. Завантаження VHI-файлів

In [1]:
import urllib.request
import os
import re
from datetime import datetime

DATA_DIR = 'vhi_data'
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = (
    'https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php'
    '?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean'
)
PROVINCE_IDS = list(range(1, 28))


def already_downloaded(pid):
    for fname in os.listdir(DATA_DIR):
        if fname.startswith(f'vhi_province_{pid}_') and fname.endswith('.csv'):
            return True
    return False


def download_vhi_files():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    headers = {'User-Agent': 'Mozilla/5.0'}
    for pid in PROVINCE_IDS:
        if already_downloaded(pid):
            print(f'[SKIP] Province {pid:2d} — вже існує')
            continue
        url = BASE_URL.format(pid=pid)
        fname = os.path.join(DATA_DIR, f'vhi_province_{pid}_{timestamp}.csv')
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=20) as resp:
                content = resp.read().decode('utf-8')
            if len(content) < 100:
                print(f'[WARN] Province {pid:2d} — порожня відповідь')
                continue
            with open(fname, 'w', encoding='utf-8') as f:
                f.write(content)
            print(f'[OK]   Province {pid:2d}')
        except Exception as e:
            print(f'[ERR]  Province {pid:2d}: {e}')


download_vhi_files()
files = sorted(os.listdir(DATA_DIR))
print(f'\nФайлів у директорії: {len(files)}')

[OK]   Province  1
[OK]   Province  2
[OK]   Province  3
[OK]   Province  4
[OK]   Province  5
[OK]   Province  6
[OK]   Province  7
[OK]   Province  8
[OK]   Province  9
[OK]   Province 10
[OK]   Province 11
[OK]   Province 12
[OK]   Province 13
[OK]   Province 14
[OK]   Province 15
[OK]   Province 16
[OK]   Province 17
[OK]   Province 18
[OK]   Province 19
[OK]   Province 20
[OK]   Province 21
[OK]   Province 22
[OK]   Province 23
[OK]   Province 24
[OK]   Province 25
[OK]   Province 26
[OK]   Province 27

Файлів у директорії: 27


### ДІАГНОСТИКА — перші рядки одного файлу
**Запустіть цю клітинку і покажіть вивід, якщо виникнуть проблеми з парсингом.**

In [2]:
DATA_DIR = 'vhi_data'
files = sorted(os.listdir(DATA_DIR))
if not files:
    print('Директорія порожня!')
else:
    sample = os.path.join(DATA_DIR, files[0])
    print(f'Файл: {files[0]}  ({os.path.getsize(sample)} байт)')
    print('=' * 70)
    with open(sample, 'r', encoding='utf-8', errors='replace') as f:
        for i, line in enumerate(f):
            if i >= 20: break
            print(f'{i:3d}: {repr(line)}')

Файл: vhi_province_10_20260411_145833.csv  (100806 байт)
  0: "Mean data for UKR  Province= 10: Khmel'nyts'kyy,  from 1981 to 2024, weekly; version='GC_current'<br>for cropland area only<br>\n"
  1: 'year,week, SMN,SMT,VCI,TCI, VHI<br>\n'
  2: '<tt><pre>1982, 1, 0.059,258.24, 51.11, 48.78, 49.95,\n'
  3: '1982, 2, 0.063,261.53, 55.89, 38.20, 47.04,\n'
  4: '1982, 3, 0.063,263.45, 57.30, 32.69, 44.99,\n'
  5: '1982, 4, 0.061,265.10, 53.96, 28.62, 41.29,\n'
  6: '1982, 5, 0.058,266.42, 46.87, 28.57, 37.72,\n'
  7: '1982, 6, 0.056,267.47, 39.55, 30.27, 34.91,\n'
  8: '1982, 7, 0.055,268.58, 35.19, 31.10, 33.14,\n'
  9: '1982, 8, 0.057,270.15, 33.35, 32.09, 32.72,\n'
 10: '1982, 9, 0.057,271.60, 30.82, 34.71, 32.77,\n'
 11: '1982,10, 0.057,273.10, 27.66, 36.79, 32.23,\n'
 12: '1982,11, 0.063,275.28, 26.28, 34.48, 30.38,\n'
 13: '1982,12, 0.074,277.68, 25.86, 36.39, 31.12,\n'
 14: '1982,13, 0.085,279.65, 22.76, 40.53, 31.65,\n'
 15: '1982,14, 0.098,281.32, 18.26, 46.96, 32.61,\n'
 16: '1982

### Пункт 3. Зчитування файлів у DataFrame та Data Cleaning

In [3]:
import pandas as pd
import numpy as np
import io
import os
import re

DATA_DIR = 'vhi_data'

NOAA_TO_NAME = {
    1: 'Cherkasy',    2: 'Chernihiv',    3: 'Chernivtsi',
    4: 'Crimea',      5: 'Dnipro',       6: 'Donetsk',
    7: 'Ivano-Frankivsk', 8: 'Kharkiv',  9: 'Kherson',
    10: 'Khmelnytskyi', 11: 'Kiev',      12: 'Kiev City',
    13: 'Kirovohrad', 14: 'Luhansk',    15: 'Lviv',
    16: 'Mykolaiv',   17: 'Odessa',     18: 'Poltava',
    19: 'Rivne',      20: 'Sevastopol', 21: 'Sumy',
    22: 'Ternopil',   23: 'Vinnytsia',  24: 'Volyn',
    25: 'Zakarpattia',26: 'Zaporizhzhia',27: 'Zhytomyr'
}

COLUMNS = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']


def load_single_file(filepath, noaa_id):
    """
    Зчитує один NOAA VHI-файл у DataFrame.

    Формат файлу:
      <head> ... </head>            ← XML-заголовок (рядки з '<')
      Year, Week, SMN, SMT, ...    ← рядок назв стовпців (може бути відсутнім)
      1981,    1,  0.00, ...       ← рядки даних (можуть мати кінцеву кому)
    """
    try:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            raw = f.read()

        # Рядки без XML-тегів та не порожні
        lines = [l for l in raw.splitlines() if l.strip() and '<' not in l]

        if not lines:
            print(f'  [WARN] Province {noaa_id}: файл не містить рядків без XML-тегів')
            return None

        # Знаходимо рядок-заголовок (містить слово 'Year')
        header_idx = None
        for idx, line in enumerate(lines):
            if re.search(r'\bYear\b', line, re.IGNORECASE):
                header_idx = idx
                break

        # Якщо заголовок знайдено — дані після нього; якщо ні — всі рядки є даними
        if header_idx is not None:
            data_lines = lines[header_idx + 1:]
        else:
            data_lines = lines

        if not data_lines:
            print(f'  [WARN] Province {noaa_id}: рядки даних відсутні')
            return None

        # Кожен рядок NOAA закінчується комою — прибираємо її
        data_str = '\n'.join(l.rstrip().rstrip(',') for l in data_lines)

        df = pd.read_csv(
            io.StringIO(data_str),
            header=None,
            names=COLUMNS,
            skipinitialspace=True,
            on_bad_lines='skip'
        )

        # Числові типи
        for col in COLUMNS:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Відкидаємо рядки без Year/Week
        df.dropna(subset=['Year', 'Week'], inplace=True)

        if df.empty:
            print(f'  [WARN] Province {noaa_id}: порожній після dropna')
            return None

        # Видаляємо sentinel-значення -1
        df = df[df['VHI'] != -1.0].copy()

        # Заповнюємо пропуски медіаною
        df.fillna(df.median(numeric_only=True), inplace=True)

        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        df['noaa_id']     = noaa_id
        df['region_name'] = NOAA_TO_NAME.get(noaa_id, f'Province_{noaa_id}')

        return df

    except Exception as e:
        print(f'  [ERR] Province {noaa_id}: {type(e).__name__}: {e}')
        return None


def load_all_files():
    frames = []
    for fname in sorted(os.listdir(DATA_DIR)):
        if not fname.endswith('.csv'):
            continue
        m = re.search(r'vhi_province_(\d+)_', fname)
        if not m:
            continue
        noaa_id = int(m.group(1))
        df = load_single_file(os.path.join(DATA_DIR, fname), noaa_id)
        if df is not None and not df.empty:
            frames.append(df)
            print(f'  Province {noaa_id:2d} ({NOAA_TO_NAME.get(noaa_id,"?")}): {len(df):4d} рядків')
        else:
            print(f'  Province {noaa_id:2d}: порожньо або помилка')

    if not frames:
        print('\nЖоден файл не зчитано. Запустіть клітинку ДІАГНОСТИКА.')
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


print('Зчитування файлів:')
vhi_raw = load_all_files()

if not vhi_raw.empty:
    print(f'\nЗагалом рядків : {len(vhi_raw):,}')
    print(f'Регіонів       : {vhi_raw["noaa_id"].nunique()}')
    print(f'Роки           : {vhi_raw["Year"].min()} – {vhi_raw["Year"].max()}')
    display(vhi_raw.head())

Зчитування файлів:
  Province 10 (Khmelnytskyi): 2185 рядків
  Province 11 (Kiev): 2185 рядків
  Province 12 (Kiev City): 2185 рядків
  Province 13 (Kirovohrad): 2185 рядків
  Province 14 (Luhansk): 2185 рядків
  Province 15 (Lviv): 2185 рядків
  Province 16 (Mykolaiv): 2185 рядків
  Province 17 (Odessa): 2185 рядків
  Province 18 (Poltava): 2185 рядків
  Province 19 (Rivne): 2185 рядків
  Province  1 (Cherkasy): 2185 рядків
  Province 20 (Sevastopol): 2185 рядків
  Province 21 (Sumy): 2185 рядків
  Province 22 (Ternopil): 2185 рядків
  Province 23 (Vinnytsia): 2185 рядків
  Province 24 (Volyn): 2185 рядків
  Province 25 (Zakarpattia): 2185 рядків
  Province 26 (Zaporizhzhia): 2185 рядків
  Province 27 (Zhytomyr): 2185 рядків
  Province  2 (Chernihiv): 2185 рядків
  Province  3 (Chernivtsi): 2185 рядків
  Province  4 (Crimea): 2185 рядків
  Province  5 (Dnipro): 2185 рядків
  Province  6 (Donetsk): 2185 рядків
  Province  7 (Ivano-Frankivsk): 2185 рядків
  Province  8 (Kharkiv): 2185 р

,Year,Week,SMN,SMT,VCI,TCI,VHI,noaa_id,region_name
0,1982,2,0.063,261.53,55.89,38.20,47.04,10,Khmelnytskyi
1,1982,3,0.063,263.45,57.30,32.69,44.99,10,Khmelnytskyi
2,1982,4,0.061,265.10,53.96,28.62,41.29,10,Khmelnytskyi
3,1982,5,0.058,266.42,46.87,28.57,37.72,10,Khmelnytskyi
4,1982,6,0.056,267.47,39.55,30.27,34.91,10,Khmelnytskyi


### Пункт 4. Заміна індексів: NOAA → українська абетка

In [4]:
UA_INDEX_MAP = {
    'Vinnytsia':        (1,  'Вінницька'),
    'Volyn':            (2,  'Волинська'),
    'Dnipro':           (3,  'Дніпропетровська'),
    'Donetsk':          (4,  'Донецька'),
    'Zhytomyr':         (5,  'Житомирська'),
    'Zakarpattia':      (6,  'Закарпатська'),
    'Zaporizhzhia':     (7,  'Запорізька'),
    'Ivano-Frankivsk':  (8,  'Івано-Франківська'),
    'Kiev':             (9,  'Київська'),
    'Kiev City':        (10, 'Київ (місто)'),
    'Kirovohrad':       (11, 'Кіровоградська'),
    'Crimea':           (12, 'Кримська АР'),
    'Luhansk':          (13, 'Луганська'),
    'Lviv':             (14, 'Львівська'),
    'Mykolaiv':         (15, 'Миколаївська'),
    'Odessa':           (16, 'Одеська'),
    'Poltava':          (17, 'Полтавська'),
    'Rivne':            (18, 'Рівненська'),
    'Sevastopol':       (19, 'Севастополь'),
    'Sumy':             (20, 'Сумська'),
    'Ternopil':         (21, 'Тернопільська'),
    'Kharkiv':          (22, 'Харківська'),
    'Kherson':          (23, 'Херсонська'),
    'Khmelnytskyi':     (24, 'Хмельницька'),
    'Cherkasy':         (25, 'Черкаська'),
    'Chernivtsi':       (26, 'Чернівецька'),
    'Chernihiv':        (27, 'Чернігівська'),
}


def remap_indices(df):
    """Додає колонки ua_id та ua_name відповідно до укр. абетки."""
    df = df.copy()
    df['ua_id']   = df['region_name'].apply(
        lambda n: UA_INDEX_MAP[n][0] if n in UA_INDEX_MAP else None
    )
    df['ua_name'] = df['region_name'].apply(
        lambda n: UA_INDEX_MAP[n][1] if n in UA_INDEX_MAP else None
    )
    return df


vhi = remap_indices(vhi_raw)

check = (
    vhi[['noaa_id', 'region_name', 'ua_id', 'ua_name']]
    .drop_duplicates()
    .sort_values('ua_id')
    .reset_index(drop=True)
)
print('Відповідність NOAA → укр. абетка:')
display(check)

unknown = vhi.loc[vhi['ua_id'].isna(), 'region_name'].unique()
if len(unknown):
    print(f'Без відповідності: {unknown}')

Відповідність NOAA → укр. абетка:


,noaa_id,region_name,ua_id,ua_name
0,23,Vinnytsia,1,Вінницька
1,24,Volyn,2,Волинська
2,5,Dnipro,3,Дніпропетровська
3,6,Donetsk,4,Донецька
4,27,Zhytomyr,5,Житомирська
5,25,Zakarpattia,6,Закарпатська
6,26,Zaporizhzhia,7,Запорізька
7,7,Ivano-Frankivsk,8,Івано-Франківська
8,11,Kiev,9,Київська
9,12,Kiev City,10,Київ (місто)


### Пункт 5а. Ряд VHI для конкретної області за вказаний рік

In [5]:
def vhi_region_year(df, ua_id, year):
    """
    Повертає ряд VHI для заданої області та року.

    Параметри:
        ua_id : int — індекс за укр. абеткою (1–27)
        year  : int — рік (1981–2024)
    """
    mask = (df['ua_id'] == ua_id) & (df['Year'] == year)
    result = (
        df.loc[mask, ['ua_name', 'Year', 'Week', 'VCI', 'TCI', 'VHI']]
        .sort_values('Week')
        .reset_index(drop=True)
    )
    return result


res = vhi_region_year(vhi, ua_id=22, year=2020)
print(f'VHI Харківська (ua_id=22), 2020 рік — {len(res)} тижнів:')
display(res)

VHI Харківська (ua_id=22), 2020 рік — 52 тижнів:


,ua_name,Year,Week,VCI,TCI,VHI
0,Харківська,2020,1,67.45,35.77,51.61
1,Харківська,2020,2,70.12,34.94,52.53
2,Харківська,2020,3,72.45,31.20,51.83
3,Харківська,2020,4,72.27,33.01,52.64
4,Харківська,2020,5,70.02,33.40,51.71
5,Харківська,2020,6,69.07,28.33,48.70
6,Харківська,2020,7,67.83,20.43,44.13
7,Харківська,2020,8,69.62,12.70,41.16
8,Харківська,2020,9,73.87,6.72,40.29
9,Харківська,2020,10,76.58,3.74,40.16


### Пункт 5б. Ряд VHI за діапазон років для вказаних областей

In [6]:
def vhi_regions_year_range(df, ua_ids, year_from, year_to):
    """
    Повертає VHI для списку областей за діапазон років.

    Параметри:
        ua_ids    : int або list[int]
        year_from : int
        year_to   : int
    """
    if isinstance(ua_ids, int):
        ua_ids = [ua_ids]
    mask = df['ua_id'].isin(ua_ids) & df['Year'].between(year_from, year_to)
    return (
        df.loc[mask, ['ua_name', 'Year', 'Week', 'VCI', 'TCI', 'VHI']]
        .sort_values(['ua_name', 'Year', 'Week'])
        .reset_index(drop=True)
    )


res = vhi_regions_year_range(vhi, ua_ids=[9, 14], year_from=2018, year_to=2020)
print(f'VHI Київська + Львівська, 2018–2020 — {len(res)} записів:')
display(res.head(20))

VHI Київська + Львівська, 2018–2020 — 312 записів:


,ua_name,Year,Week,VCI,TCI,VHI
0,Київська,2018,1,48.85,32.79,40.82
1,Київська,2018,2,51.39,35.91,43.65
2,Київська,2018,3,51.63,45.62,48.62
3,Київська,2018,4,52.67,50.88,51.77
4,Київська,2018,5,47.56,56.61,52.08
5,Київська,2018,6,40.08,63.15,51.61
6,Київська,2018,7,35.00,63.88,49.43
7,Київська,2018,8,32.52,62.73,47.62
8,Київська,2018,9,32.26,61.94,47.09
9,Київська,2018,10,31.01,62.05,46.52


### Пункт 5в. Екстремуми, середнє та медіана VHI

In [7]:
def vhi_stats(df, ua_ids=None, year_from=None, year_to=None):
    """
    Обчислює min, max, mean, median VHI для вказаних областей і діапазону років.

    Параметри:
        ua_ids    : int, list[int] або None (всі регіони)
        year_from : int | None
        year_to   : int | None
    """
    mask = pd.Series(True, index=df.index)
    if ua_ids is not None:
        if isinstance(ua_ids, int):
            ua_ids = [ua_ids]
        mask &= df['ua_id'].isin(ua_ids)
    if year_from is not None:
        mask &= df['Year'] >= year_from
    if year_to is not None:
        mask &= df['Year'] <= year_to

    return (
        df.loc[mask]
        .groupby(['ua_id', 'ua_name'])['VHI']
        .agg(
            min_vhi='min',
            max_vhi='max',
            mean_vhi=lambda x: round(x.mean(), 3),
            median_vhi='median'
        )
        .reset_index()
        .sort_values('ua_id')
    )


res = vhi_stats(vhi, year_from=2015, year_to=2024)
print('Статистика VHI по всіх регіонах (2015–2024):')
display(res)

Статистика VHI по всіх регіонах (2015–2024):


,ua_id,ua_name,min_vhi,max_vhi,mean_vhi,median_vhi
0,1,Вінницька,31.60,71.69,49.861,49.020
1,2,Волинська,19.94,71.59,46.460,46.330
2,3,Дніпропетровська,19.00,77.54,46.262,45.330
3,4,Донецька,17.88,75.78,44.208,42.870
4,5,Житомирська,30.24,65.33,47.251,46.785
5,6,Закарпатська,33.67,70.24,51.079,50.565
6,7,Запорізька,18.27,82.61,43.335,42.245
7,8,Івано-Франківська,28.30,67.90,49.108,48.040
8,9,Київська,20.91,65.72,45.221,44.445
9,10,Київ (місто),24.95,62.30,45.540,45.680


### Пункт 5г. Тижні з екстремальною посухою (VHI < 15)

In [8]:
def drought_weeks(df, year, vhi_threshold=15):
    """
    Знаходить тижні з екстремальною посухою (VHI < threshold) за вказаний рік.

    Параметри:
        year          : int
        vhi_threshold : float (15 — екстремальна посуха за NOAA)
    """
    year_df = df[df['Year'] == year]

    if year_df.empty:
        print(f'Дані за {year} рік відсутні!')
        return pd.DataFrame(columns=['ua_id','ua_name','drought_weeks','total_weeks','drought_pct'])

    # Агрегація без include_groups (сумісно з усіма версіями pandas)
    records = []
    for (uid, uname), g in year_df.groupby(['ua_id', 'ua_name']):
        drought = int((g['VHI'] < vhi_threshold).sum())
        total   = len(g)
        pct     = round(100.0 * drought / total, 1) if total else 0.0
        records.append({
            'ua_id':         uid,
            'ua_name':       uname,
            'drought_weeks': drought,
            'total_weeks':   total,
            'drought_pct':   pct,
        })

    result = (
        pd.DataFrame(records)
        .sort_values('drought_pct', ascending=False)
        .reset_index(drop=True)
    )
    return result


res = drought_weeks(vhi, year=2003)
affected = res[res['drought_weeks'] > 0]
print(f'Тижні екстремальної посухи (VHI < 15), 2003 рік:')
if not affected.empty:
    display(affected)
else:
    print('Посух не зафіксовано або дані відсутні.')

Тижні екстремальної посухи (VHI < 15), 2003 рік:


,ua_id,ua_name,drought_weeks,total_weeks,drought_pct
0,23,Херсонська,1,51,2.0
